# PML · Lecture 9 — Generative Classifiers, end to end

A **generative** classifier learns *how the data was generated* — for each class, a little story `p(x | y)` — and then runs that story **backwards with Bayes' rule** to classify. This notebook walks the whole lecture, one small visual step at a time:

1. the **generative story** (sample data from `p(y)·p(x | y)`),
2. **generative vs. discriminative** and the Bayes inversion,
3. **maximum likelihood** = just *count, average, spread* (checked on a tiny by-hand example),
4. **fit GDA** in NumPy and *see* the class-conditional Gaussians,
5. **classify** with Bayes → decision regions and the (curved) **QDA** boundary,
6. share the covariance → the boundary **straightens** into **LDA** (a line),
7. **Naive Bayes** as the independent-features special case,
8. **generative vs. discriminative** head-to-head (vs. logistic regression; who wins on little data),
9. bonus: a fitted generative model can **dream up new data**.

Pure `numpy` + `matplotlib` (+ `scikit-learn` for one comparison) — all preinstalled in Colab. Run top to bottom.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
rng = np.random.default_rng(1)
plt.rcParams['figure.figsize'] = (6.2, 5)
C0, C1 = '#1f77b4', '#d62728'          # class colours used throughout

def scatter_classes(X, y, ax=None):
    ax = ax or plt.gca()
    for k, c in [(0, C0), (1, C1)]:
        ax.scatter(*X[y==k].T, s=16, c=c, alpha=.7, edgecolor='none', label=f'class {k}')
    ax.set_xlabel('x1'); ax.set_ylabel('x2'); ax.legend(loc='upper left')
    return ax

## 1. The generative story — make some data

The model says each point is born in two steps: **(1)** pick a class `y` from a prior `p(y)`, **(2)** draw its features `x` from that class's Gaussian `p(x | y) = 𝒩(μ_y, Σ_y)`. Let's *be* the generative model and sample a dataset — two tilted blobs with **different shapes**.

In [ ]:
# the TRUE generating parameters (we'll pretend not to know them and re-estimate later)
pi_true = [0.5, 0.5]
mu_true = {0: np.array([2., 2.]), 1: np.array([5., 4.])}
Sig_true = {0: np.array([[1.2, 0.5], [0.5, 0.6]]),      # class 0: tilted one way
            1: np.array([[0.6, -0.4], [-0.4, 1.2]])}     # class 1: tilted the other

def sample(n):
    y = rng.choice([0, 1], size=n, p=pi_true)            # step 1: draw the class
    X = np.array([rng.multivariate_normal(mu_true[k], Sig_true[k]) for k in y])  # step 2: draw x | y
    return X, y

X, y = sample(300)
scatter_classes(X, y); plt.title('Data sampled from p(y)·p(x | y)'); plt.show()
print('class counts:', {k: int((y==k).sum()) for k in (0, 1)})

That's the whole generative idea in code: **a prior over classes, and one Gaussian per class.** Everything below is either *fitting* that story to data or *inverting* it to classify.

### The picture — a directed graphical model

That two-step story *is* a tiny **directed graphical model**. For each example `n`, the class `yₙ` is drawn from the prior, then the features `xₙ` are drawn from that class's Gaussian; the plate repeats for all `N` points:

```
   π ──▶ (yₙ) ──▶ (xₙ) ◀── θ = {μ_k, Σ_k}
                 └──── plate:  n = 1 … N ────┘
```

It encodes the factorization `p(x, y) = p(y)·p(x | y)`. The circles are the random variables (`yₙ`, `xₙ`); `π` (the class prior) and `θ` (the class means and covariances) are the **parameters** we'll estimate. Classifying is running the arrow **backwards** — from an observed `x`, infer `y` — with Bayes' rule.

## 2. Generative vs. discriminative — and Bayes' rule

- A **discriminative** model learns the boundary `p(y | x)` directly (logistic regression, nets).
- A **generative** model learns the *whole story* `p(x, y) = p(y)·p(x | y)`, then flips it with **Bayes' rule** to classify:

$$p(y{=}k \mid x) = \frac{\pi_k\,\mathcal{N}(x;\mu_k,\Sigma_k)}{\sum_j \pi_j\,\mathcal{N}(x;\mu_j,\Sigma_j)}.$$

To *decide*, we don't even need the denominator — compare the numerators (in log space, the **score** `δ_k(x)`) and take the biggest. Generative models assume more (a full model of `x`), so they can squeeze more out of **little data**; discriminative models assume less. We'll see both facts for real below.

## 3. Fitting by maximum likelihood = *count, average, spread*

The ML estimates have no gradient descent — one pass over the data:

$$\hat\pi_k = \frac{N_k}{N},\qquad \hat\mu_k = \frac{1}{N_k}\!\sum_{n:y_n=k} x_n,\qquad \hat\Sigma_k = \frac{1}{N_k}\!\sum_{n:y_n=k}(x_n-\hat\mu_k)(x_n-\hat\mu_k)^\top.$$

**Priors are class frequencies, means are class averages, covariances are class spreads.** Let's check that on the lecture's tiny by-hand example first, where we know the answer.

In [ ]:
# lecture's tiny dataset: class 0 = (1,2),(2,0),(3,1); class 1 = (6,5),(7,7),(8,6)
Xt = np.array([[1,2],[2,0],[3,1],[6,5],[7,7],[8,6]], float)
yt = np.array([0,0,0,1,1,1])

for k in (0, 1):
    Xk = Xt[yt==k]
    mu = Xk.mean(0)
    d  = Xk - mu
    Sig = (d.T @ d) / len(Xk)          # average outer product = count, average, spread
    print(f'class {k}:  pi={len(Xk)/len(yt):.2f}  mu={mu}  Sigma=\n{np.round(3*Sig,2)}/3')
# matches the lecture by hand: mu0=(2,1), mu1=(7,6), Sigma0=[[2,-1],[-1,2]]/3, Sigma1=[[2,1],[1,2]]/3

The code reproduces the hand calculation exactly (`μ₀=(2,1)`, `Σ₀=⅓[[2,−1],[−1,2]]`, …). No magic — the MLE really is just counting and averaging.

## 4. Fit GDA (QDA) in NumPy

The same three lines, wrapped up. `fit_gda` returns the priors, means and **per-class** covariances; `log_gaussian` is the log-density; `predict` scores every class and takes the argmax — Bayes' rule.

In [ ]:
def fit_gda(X, y):
    classes = np.unique(y); N = len(y)
    priors, means, covs = {}, {}, {}
    for k in classes:
        Xk = X[y==k]
        priors[k] = len(Xk)/N                    # pi_k = N_k / N
        means[k]  = Xk.mean(0)                    # mu_k = class mean
        d = Xk - means[k]
        covs[k]   = (d.T @ d)/len(Xk)             # Sigma_k = within-class covariance
    return classes, priors, means, covs

def log_gaussian(x, mu, Sigma):
    d = x - mu
    _, logdet = np.linalg.slogdet(Sigma)
    return -0.5*(logdet + d @ np.linalg.solve(Sigma, d) + len(mu)*np.log(2*np.pi))

classes, priors, means, covs = fit_gda(X, y)
for k in classes:
    print(f'class {k}:  pi={priors[k]:.2f}  mu={np.round(means[k],2)}')
print('recovered means are close to the TRUE', {k: mu_true[k].tolist() for k in (0,1)}, '- fitting works')

### Classify one new point — Bayes' rule by hand

Scoring is `δ_k(x) = log π_k + log 𝒩(x; μ_k, Σ_k)`; the biggest score wins. Take the lecture's tiny example with the shared covariance `Σ = ⅔·I` and classify `x = (5, 4)`:

In [ ]:
mu_tiny = {0: np.array([2., 1.]), 1: np.array([7., 6.])}
Sig_tiny = (2/3) * np.eye(2)                      # the pooled LDA covariance from the by-hand fit
x_new = np.array([5., 4.])
for k in (0, 1):
    delta = np.log(0.5) + log_gaussian(x_new, mu_tiny[k], Sig_tiny)
    print(f'delta_{k}(x) = {delta:.2f}')
print('-> argmax picks class 1 (x=(5,4) is on the x1+x2>8 side of the line)')

## 5. See the class-conditionals — one Gaussian per class

A generative classifier literally draws a **blob** for each class. Here are the fitted Gaussians as 1σ/2σ ellipses on top of the data.

In [ ]:
def gaussian_ellipses(ax, mu, Sigma, color):
    vals, vecs = np.linalg.eigh(Sigma)
    ang = np.degrees(np.arctan2(*vecs[:, 1][::-1]))
    for s in (1, 2):                              # 1-sigma and 2-sigma contours
        w, h = 2*s*np.sqrt(vals)
        ax.add_patch(Ellipse(mu, w, h, angle=ang, fill=False, edgecolor=color, lw=1.6, alpha=.8))

ax = scatter_classes(X, y)
for k, c in [(0, C0), (1, C1)]:
    gaussian_ellipses(ax, means[k], covs[k], c)
    ax.scatter(*means[k], c=c, marker='X', s=120, edgecolor='k', zorder=5)
plt.title('Fitted class-conditional Gaussians (X = class mean)'); plt.show()

## 6. Classify with Bayes → decision regions (QDA is *curved*)

To classify a point, score each class with
$$\delta_k(x) = \log\pi_k - \tfrac12\log|\Sigma_k| - \tfrac12 (x-\mu_k)^\top\Sigma_k^{-1}(x-\mu_k)$$
and take the argmax. Because each class keeps **its own** `Σ_k`, that squared term is quadratic in `x`, so the boundary **bends**. Let's paint the decision regions.

In [ ]:
def class_scores(G, classes, priors, means, covs):
    """delta_k(x) for a whole grid G (rows = points), vectorised."""
    S = np.empty((len(G), len(classes)))
    for j, k in enumerate(classes):
        d = G - means[k]
        _, logdet = np.linalg.slogdet(covs[k])
        quad = np.einsum('ni,ij,nj->n', d, np.linalg.inv(covs[k]), d)
        S[:, j] = np.log(priors[k]) - 0.5*logdet - 0.5*quad
    return S

def plot_regions(classes, priors, means, covs, title):
    xx, yy = np.meshgrid(np.linspace(-1, 8, 300), np.linspace(-1, 7, 300))
    G = np.c_[xx.ravel(), yy.ravel()]
    pred = classes[class_scores(G, classes, priors, means, covs).argmax(1)].reshape(xx.shape)
    ax = plt.gca()
    ax.contourf(xx, yy, pred, levels=[-.5,.5,1.5], colors=[C0, C1], alpha=.12)
    ax.contour(xx, yy, pred, levels=[.5], colors='k', linewidths=2)   # the boundary
    scatter_classes(X, y, ax); ax.set_title(title)

plot_regions(classes, priors, means, covs, 'QDA — per-class Σ → curved boundary'); plt.show()

## 7. The soft answer — posterior probability heatmap

Bayes gives a full **probability**, not just a label. `p(y=1 | x)` is a `softmax` of the scores; near the boundary it's ~0.5 (the model is unsure), and it saturates to 0/1 out in each blob.

In [ ]:
xx, yy = np.meshgrid(np.linspace(-1, 8, 300), np.linspace(-1, 7, 300))
G = np.c_[xx.ravel(), yy.ravel()]
S = class_scores(G, classes, priors, means, covs)
p1 = (np.exp(S - S.max(1, keepdims=True))[:, 1] /
      np.exp(S - S.max(1, keepdims=True)).sum(1)).reshape(xx.shape)   # softmax -> P(y=1|x)
im = plt.contourf(xx, yy, p1, levels=np.linspace(0, 1, 11), cmap='coolwarm', alpha=.8)
plt.colorbar(im, label='P(y=1 | x)')
plt.contour(xx, yy, p1, levels=[.5], colors='k', linewidths=2)
for k, c in [(0, C0), (1, C1)]:                    # black edges so points read on the heatmap
    plt.scatter(*X[y==k].T, s=14, c=c, edgecolor='k', linewidth=.3, label=f'class {k}')
plt.legend(loc='upper left'); plt.title('Posterior P(y=1 | x)'); plt.show()

## 8. Share the covariance → the boundary straightens (LDA)

Now force **one shared `Σ`** across classes (pool the within-class scatter). The quadratic `xᵀΣ⁻¹x` term becomes identical for every class and **cancels** when comparing them — what's left is *linear*, so the boundary is a straight line. That's **LDA**, and it's the exact form logistic regression learns.

In [ ]:
def fit_lda(X, y):
    classes = np.unique(y); N = len(y)
    priors, means = {}, {}
    S = np.zeros((X.shape[1], X.shape[1]))
    for k in classes:
        Xk = X[y==k]; priors[k] = len(Xk)/N; means[k] = Xk.mean(0)
        d = Xk - means[k]; S += d.T @ d               # pool the scatter
    Sigma = S / N                                      # one shared covariance
    covs = {k: Sigma for k in classes}                # same Σ for every class
    return classes, priors, means, covs

lda = fit_lda(X, y)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plt.sca(axes[0]); plot_regions(classes, priors, means, covs, 'QDA (per-class Σ) — curved')
plt.sca(axes[1]); plot_regions(*lda, 'LDA (shared Σ) — straight line')
plt.tight_layout(); plt.show()

Same data, same generative recipe — the **only** change is per-class vs. shared `Σ`, and the fence goes from curved to straight. (The little loop in the QDA corner is real: a quadratic boundary is a *conic*, so far from the data it can curve back — a region with no points and no practical effect.)

### Read the line off directly: `w = Σ⁻¹μ`

With a shared `Σ`, each score is *linear*: `δ_k(x) = w_kᵀx + b_k` with `w_k = Σ⁻¹μ_k` and `b_k = log π_k − ½ μ_kᵀΣ⁻¹μ_k`. For two classes the boundary is `δ₁ − δ₀ = 0`, i.e. `wᵀx + b = 0` with `w = w₁ − w₀`. Let's compute it from the fitted LDA and draw that line — it lands exactly on the boundary from the contour plot.

In [ ]:
cs, pr, mu, cv = lda
Si = np.linalg.inv(cv[0])                                   # shared Σ⁻¹
w = {k: Si @ mu[k] for k in cs}
b = {k: np.log(pr[k]) - 0.5 * mu[k] @ Si @ mu[k] for k in cs}
w2, b2 = w[1] - w[0], b[1] - b[0]                           # two-class line  w2·x + b2 = 0
print('w =', np.round(w2, 2), '  b =', round(float(b2), 2))

ax = scatter_classes(X, y)
xs = np.array([-1., 8.]); ax.plot(xs, -(w2[0]*xs + b2)/w2[1], 'k', lw=2)
ax.set_title('Boundary from w = Σ⁻¹μ  (same line as the LDA fence)'); plt.show()

## 9. Naive Bayes — assume the features are independent given the class

**Naive Bayes** is GDA with a **diagonal** covariance: within a class, it treats each feature as independent, so the class-conditional factorizes into one 1-D Gaussian per feature, `p(x | y) = ∏_j p(x_j | y)`. Cheap and often surprisingly good — the *decision* frequently survives the wrong independence assumption. Its blobs are axis-aligned, so its boundary ignores the correlation.

In [ ]:
def fit_naive_bayes(X, y):
    classes, priors, means, covs = fit_gda(X, y)
    covs = {k: np.diag(np.diag(covs[k])) for k in classes}   # keep only the diagonal
    return classes, priors, means, covs

plot_regions(*fit_naive_bayes(X, y), 'Naive Bayes — diagonal Σ (axis-aligned blobs)'); plt.show()

## 10. Generative vs. discriminative, head to head

LDA (generative) and **logistic regression** (discriminative) fit the *same shape* of boundary but in opposite ways: LDA models `p(x | y)` then derives the line; logistic regression fits the line to maximise `p(y | x)` directly — same shape, two philosophies.

They also differ in **how much data they need**. The classic result (Ng & Jordan): a generative model like **Naive Bayes** has few parameters and reaches good accuracy with *fewer* examples, while **logistic regression** needs more data but catches up (and can overtake once there's enough). Let's watch it — a higher-dimensional problem where Naive Bayes is well specified, trained at growing sizes.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB          # = the diagonal-Σ generative model from §9

D, gap = 40, 0.42                                   # 40 independent features, modest separation
muhd = {0: np.zeros(D), 1: np.full(D, gap)}
def sample_hd(n):
    yy = rng.integers(0, 2, n)
    XX = np.array([rng.normal(muhd[k], 1.0) for k in yy])
    return XX, yy

Xte_hd, yte_hd = sample_hd(4000)
sizes = [40, 80, 160, 320, 640, 1280]
nb_acc, lr_acc = [], []
for n in sizes:
    a_nb, a_lr = [], []
    for _ in range(40):
        Xtr, ytr = sample_hd(n)
        if len(np.unique(ytr)) < 2:                 # guard the rare single-class draw
            continue
        a_nb.append((GaussianNB().fit(Xtr, ytr).predict(Xte_hd) == yte_hd).mean())
        a_lr.append((LogisticRegression(C=1e6, max_iter=3000)      # large C = ~unregularized
                     .fit(Xtr, ytr).predict(Xte_hd) == yte_hd).mean())
    nb_acc.append(np.mean(a_nb)); lr_acc.append(np.mean(a_lr))

plt.figure(figsize=(6.5, 4.2))
plt.plot(sizes, nb_acc, 'o-', label='Naive Bayes (generative)')
plt.plot(sizes, lr_acc, 's-', label='Logistic regression (discriminative)')
plt.xscale('log'); plt.xlabel('training-set size (log)'); plt.ylabel('test accuracy')
plt.legend(); plt.title(f'Who needs less data? {D}-D problem (avg of 40 runs)'); plt.show()

The generative model (Naive Bayes) is **ahead in the small-data regime** — its independent-feature assumption is extra information that pays off when examples are scarce — and the discriminative model **catches up as data grows** (and would overtake if the assumption were wrong). That's the trade-off in one picture: *assume more and win early, or assume less and win late.* Same decision, two philosophies — the theme of the whole lecture.

## 11. Bonus — a generative model can *dream up new data*

Because it modelled `p(x, y)`, we can run the story forward and **sample brand-new points** — something a discriminative model simply cannot do. Fit, then generate:

In [ ]:
cs, pr, mu, cv = fit_gda(X, y)
yg = rng.choice(cs, size=300, p=[pr[k] for k in cs])
Xg = np.array([rng.multivariate_normal(mu[k], cv[k]) for k in yg])
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
scatter_classes(X, y, ax[0]); ax[0].set_title('real data')
scatter_classes(Xg, yg, ax[1]); ax[1].set_title('data dreamed up by the fitted model')
plt.show()

## Your turn

1. **Priors matter.** Re-sample with `pi_true = [0.9, 0.1]` and refit. How does the decision boundary shift, and why? (Look at the `log π_k` term in `δ_k`.)
2. **QDA vs LDA when shapes differ.** Make the two classes' covariances very different and compare QDA and LDA accuracy on a fresh test set — when does the curve beat the line?
3. **Naive Bayes cost.** Give the classes a strong correlation (large off-diagonal `Σ`). How much accuracy does Naive Bayes lose versus full QDA?
4. **Classify by hand, check by code.** For the tiny dataset in §3 with shared `Σ = ⅔I`, classify `x = (5, 4)` with `δ_k`; confirm it lands in class 1 and on the `x₁+x₂ > 8` side of the line.
5. **Three classes.** Add a third blob and confirm nothing changes but the loop range — argmax over `δ_k` handles any number of classes.

## Recap
- A generative classifier = **a prior `p(y)` + one class-conditional `p(x | y)` per class**, inverted with **Bayes' rule**.
- Fitting is **count, average, spread** — closed-form ML for `π, μ, Σ`.
- **QDA** (per-class `Σ`) → curved boundary; **LDA** (shared `Σ`) → a line = the logistic form; **Naive Bayes** = diagonal `Σ`.
- Generative models assume more, so they **win on little data** and can **generate** — the trade-off against discriminative models like logistic regression (next lecture).